In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatGroq(model='llama-3.3-70b-versatile')

In [3]:
# define state
class JokeState(TypedDict):
    topic: str
    joke: str
    explanantion: str
    

In [4]:
def generate_joke(state: JokeState):
    
    prompt = f'Generate a joke for the topic: {state['topic']}'
    response = llm.invoke(prompt).content
    
    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):
    
    prompt = f'Genearte explanation from the following joke: {state['joke']}'
    response = llm.invoke(prompt).content
    
    return {'explanantion': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {'configurable': {'thread_id': '1'}}
workflow.invoke({'topic':'pizza'} ,config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanantion': 'The joke is a play on words. The phrase "feeling a little crusty" has a double meaning here. \n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and crunchy. \n\nHowever, "crusty" can also be used to describe someone who is irritable, grumpy, or in a bad mood. \n\nSo, the joke is saying that the pizza is in a bad mood because it\'s feeling a little "crusty", which is a clever pun that connects the physical characteristic of the pizza (its crust) to its emotional state (being grumpy). The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}

In [8]:
config2 = {'configurable': {'thread_id': '2'}}
workflow.invoke({'topic':'pasta'} ,config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.',
 'explanantion': 'Let\'s break down this joke and explain why it\'s funny.\n\nThe joke is a play on words, using a common phrase "tangled up in a relationship" and giving it a literal twist. In this case, the spaghetti is afraid of getting married because it\'s afraid of getting "tangled up" - which is a common phrase used to describe the complexities and challenges of a romantic relationship.\n\nHowever, spaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. So, in this joke, the phrase "tangled up" has a double meaning. The spaghetti is not just afraid of the emotional complexities of a relationship, but also literally afraid of getting tangled up with its partner, like the way spaghetti can get tangled up on a plate.\n\nThis joke relies on a clever use of language, using a common phrase in a new and un

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanantion': 'The joke is a play on words. The phrase "feeling a little crusty" has a double meaning here. \n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and crunchy. \n\nHowever, "crusty" can also be used to describe someone who is irritable, grumpy, or in a bad mood. \n\nSo, the joke is saying that the pizza is in a bad mood because it\'s feeling a little "crusty", which is a clever pun that connects the physical characteristic of the pizza (its crust) to its emotional state (being grumpy). The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16f0ad-5f71-65eb-8002-fd273b6fc21f'}}, metadata={'source': 'loop', 'step': 2, 'parents':

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanantion': 'The joke is a play on words. The phrase "feeling a little crusty" has a double meaning here. \n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and crunchy. \n\nHowever, "crusty" can also be used to describe someone who is irritable, grumpy, or in a bad mood. \n\nSo, the joke is saying that the pizza is in a bad mood because it\'s feeling a little "crusty", which is a clever pun that connects the physical characteristic of the pizza (its crust) to its emotional state (being grumpy). The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16f0ad-5f71-65eb-8002-fd273b6fc21f'}}, metadata={'source': 'loop', 'step': 2, 'parents'

TIME TRAVEL

In [11]:
workflow.get_state({'configurable': {'thread_id': '1', 'checkpoint_id':'1f16ed6e-da68-69b0-8000-4b3e54b3cf11'}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f16ed6e-da68-69b0-8000-4b3e54b3cf11'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())